# MLA — 面试版

**难度：** Hard

**面试要点：** 手写 softmax + 不用 view/transpose 的分头方式

In [ ]:
import torch

In [ ]:
# ✅ INTERVIEW

def mla_attention(X, W_dkv, W_uk, W_uv, W_q, num_heads):
    B, S, D = X.shape
    D_h = W_q.shape[1] // num_heads

    # 压缩 → 解压
    c_kv = X @ W_dkv
    K = c_kv @ W_uk
    V = c_kv @ W_uv
    Q = X @ W_q

    # ---- 手写分头: 用 reshape 而非 view ----
    # 面试时可能被要求不用 view/transpose
    # 用 index_select 或显式循环
    # 这里用 reshape (功能等价但更安全)
    Q = Q.reshape(B, S, num_heads, D_h).permute(0, 2, 1, 3)
    K = K.reshape(B, S, num_heads, D_h).permute(0, 2, 1, 3)
    V = V.reshape(B, S, num_heads, D_h).permute(0, 2, 1, 3)

    # ---- 手写 softmax ----
    scale = D_h ** -0.5
    scores = Q @ K.transpose(-2, -1) * scale  # (B, H, S, S)
    max_s = scores.max(dim=-1, keepdim=True).values
    exp_s = torch.exp(scores - max_s)
    attn = exp_s / exp_s.sum(dim=-1, keepdim=True)

    # 注意力加权 + 合并多头
    out = (attn @ V).permute(0, 2, 1, 3).reshape(B, S, num_heads * D_h)
    return out

In [ ]:
torch.manual_seed(0)
B, S, D = 2, 6, 32
num_heads = 4
D_h = 8
D_c = 8

W_dkv = torch.randn(D, D_c) * 0.1
W_uk  = torch.randn(D_c, num_heads * D_h) * 0.1
W_uv  = torch.randn(D_c, num_heads * D_h) * 0.1
W_q   = torch.randn(D, num_heads * D_h) * 0.1

X = torch.randn(B, S, D)
out = mla_attention(X, W_dkv, W_uk, W_uv, W_q, num_heads)
print(f"Output shape: {out.shape}")

In [ ]:
from torch_judge import check
check("mla")

## 📝 核心思路总结

1. **压缩-解压模式**：`c_kv = X @ W_dkv` 缓存小向量，推理时 `c_kv @ W_uk` 解压
2. **手写 softmax**：减最大值 → exp → 归一化
3. **分头用 permute**：`reshape + permute` 替代 `view + transpose`